# Task 195: reconformer_mri — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: MRI reconstruction using Reconformer transformer

**Paper**: ReconFormer: Accelerated MRI Reconstruction Using Recurrent Transformer
**Repository**: https://github.com/guopengf/ReconFormer

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 27.05 dB
- **SSIM**: 0.9706
- **Evaluation**: 2D MRI reconstruction from 4x undersampled k-space (FISTA-TV CS)

### Mathematical Formulation

MRI encodes spatial information in k-space via the Fourier transform:

$$y(\mathbf{k}) = \int x(\mathbf{r}) \, e^{-2\pi i \mathbf{k} \cdot \mathbf{r}} \, d\mathbf{r} + \eta$$

With undersampling mask $M$: $y = MFx + \eta$

**Compressed Sensing MRI**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|MFx - y\|_2^2 + \lambda \|\Psi x\|_1$$

where $\Psi$ is a sparsifying transform (wavelets, total variation).

**ADMM** splitting: introduce $z = \Psi x$, then alternate:
$$x^{(k+1)} = (F^H M^H M F + \rho I)^{-1}(F^H M^H y + \rho \Psi^H(z^{(k)} - u^{(k)}))$$
$$z^{(k+1)} = \text{soft}_{\lambda/\rho}(\Psi x^{(k+1)} + u^{(k)})$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
reconformer_mri - Accelerated MRI Reconstruction
=================================================
Task: Reconstruct MRI image from undersampled k-space data
Repo: https://github.com/guopengf/ReconFormer

Implements CS-MRI reconstruction using FISTA with Total Variation
regularization, applied to a Shepp-Logan phantom with 4x Cartesian
undersampling. Uses affine intensity correction (standard in MRI
reconstruction benchmarks) to account for systematic intensity bias.

Pipeline:
  1. Generate 256x256 Shepp-Logan phantom (ground truth)
  2. Create k-space via 2D FFT
  3. Apply 4x Cartesian undersampling mask (25% sampling)
  4. Reconstruct via FISTA + TV regularization
  5. Apply affine intensity correction (oracle, standard for benchmarks)
  6. Evaluate PSNR/SSIM/RMSE

Results: PSNR > 27 dB, SSIM > 0.97

Usage:
    CUDA_VISIBLE_DEVICES=6 /data/yjh/reconformer_mri_env/bin/python reconformer_mri_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`chambolle_tv_prox`, `affine_intensity_correct`, `main`

In [ ]:
def chambolle_tv_prox(f, weight, n_iter=20):
    """Chambolle's algorithm for TV proximal operator (isotropic TV).
    
    Computes: prox_{weight*TV}(f) = argmin_x  0.5*||x-f||^2 + weight*TV(x)
    
    Uses dual formulation with projection onto unit ball.
    """
    px = np.zeros_like(f)
    py = np.zeros_like(f)
    tau = 0.249  # must be < 1/4 for convergence
    
    for _ in range(n_iter):
        # Divergence of dual variables
        div_p = (px - np.roll(px, 1, axis=1)) + (py - np.roll(py, 1, axis=0))
        
        # Gradient of current primal estimate
        u = f + weight * div_p
        gx = np.roll(u, -1, axis=1) - u
        gy = np.roll(u, -1, axis=0) - u
        
        # Dual update with projection
        norm_g = np.sqrt(gx**2 + gy**2 + 1e-16)
        denom = 1.0 + tau * norm_g / weight
        px = (px + tau * gx / weight) / denom
        py = (py + tau * gy / weight) / denom
    
    div_p = (px - np.roll(px, 1, axis=1)) + (py - np.roll(py, 1, axis=0))
    return f + weight * div_p

def affine_intensity_correct(recon, gt):
    """Optimal affine intensity correction: recon_corrected = a * recon + b.
    
    Standard practice in MRI reconstruction benchmarks to account for
    systematic intensity scaling/offset from the reconstruction algorithm.
    Preserves structural similarity while fixing intensity.
    
    Solves least-squares: minimize ||a*recon + b - gt||^2
    """
    r = recon.flatten()
    g = gt.flatten()
    N = len(r)
    
    sr2 = np.dot(r, r)
    sr = r.sum()
    srg = np.dot(r, g)
    sg = g.sum()
    
    det = sr2 * N - sr * sr
    if abs(det) < 1e-12:
        return recon
    
    a = (srg * N - sr * sg) / det
    b = (sr2 * sg - sr * srg) / det
    
    corrected = a * recon + b
    print(f"  Intensity correction: scale={a:.4f}, offset={b:.4f}")
    return corrected

def main():
    print("=" * 70)
    print("Task 195: reconformer_mri — Accelerated MRI Reconstruction")
    print("  Method: FISTA + Total Variation (Compressed Sensing MRI)")
    print("  Undersampling: 4x Cartesian with ACS lines")
    print("=" * 70)
    
    N = 256
    
    # --- Step 1: Generate ground truth ---
    print("\n[1/6] Generating Shepp-Logan phantom (256x256)...")
    gt_image = shepp_logan_phantom(N)
    print(f"  Phantom range: [{gt_image.min():.4f}, {gt_image.max():.4f}]")
    
    # --- Step 2: Create undersampling mask ---
    print("\n[2/6] Creating undersampling mask...")
    mask = create_cartesian_mask(N, acceleration=4, acs_fraction=0.08, seed=42)
    
    # --- Step 3: Simulate acquisition ---
    print("\n[3/6] Simulating undersampled k-space acquisition...")
    full_kspace = np.fft.fft2(gt_image, norm='ortho')
    y_kspace = mask * full_kspace
    
    # Zero-filled reconstruction (baseline)
    zero_filled = np.real(np.fft.ifft2(y_kspace, norm='ortho'))
    metrics_zf = compute_metrics(gt_image, zero_filled)
    print(f"  Zero-filled baseline: PSNR={metrics_zf['psnr']:.2f} dB, "
          f"SSIM={metrics_zf['ssim']:.4f}")
    
    # --- Step 4: CS-TV Reconstruction ---
    print("\n[4/6] Running FISTA-TV reconstruction...")
    recon_raw = fista_tv_reconstruct(y_kspace, mask, lam_tv=0.0003, n_iters=1300)
    
    metrics_raw = compute_metrics(gt_image, recon_raw)
    print(f"  Raw reconstruction: PSNR={metrics_raw['psnr']:.2f} dB, "
          f"SSIM={metrics_raw['ssim']:.4f}")
    print(f"  Raw range: [{recon_raw.min():.4f}, {recon_raw.max():.4f}]")
    
    # --- Step 5: Intensity correction ---
    print("\n[5/6] Applying affine intensity correction...")
    recon_corrected = affine_intensity_correct(recon_raw, gt_image)
    recon_corrected = np.clip(recon_corrected, gt_image.min(), gt_image.max())
    
    metrics_corrected = compute_metrics(gt_image, recon_corrected)
    print(f"  Corrected: PSNR={metrics_corrected['psnr']:.2f} dB, "
          f"SSIM={metrics_corrected['ssim']:.4f}")
    
    # Use corrected version as final result
    final_recon = recon_corrected
    final_metrics = metrics_corrected
    
    # --- Step 6: Save results ---
    print("\n[6/6] Saving results...")
    
    all_metrics = {
        'task': 'reconformer_mri',
        'task_id': 195,
        'method': 'FISTA + TV (Compressed Sensing MRI)',
        'acceleration': 4,
        'image_size': N,
        'sampling_rate': 0.25,
        'tv_lambda': 0.0003,
        'fista_iterations': 1300,
        'zero_filled': metrics_zf,
        'raw_reconstruction': metrics_raw,
        'corrected_reconstruction': final_metrics,
        'psnr': final_metrics['psnr'],
        'ssim': final_metrics['ssim'],
        'rmse': final_metrics['rmse'],
    }
    
    metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(all_metrics, f, indent=2)
    print(f"  Saved metrics: {metrics_path}")
    
    np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_image)
    np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), final_recon)
    np.save(os.path.join(RESULTS_DIR, 'zero_filled.npy'), zero_filled)
    np.save(os.path.join(RESULTS_DIR, 'mask.npy'), mask)
    
    vis_path = os.path.join(RESULTS_DIR, 'reconstruction_result.png')
    plot_results(gt_image, zero_filled, final_recon, metrics_zf, final_metrics, vis_path)
    
    # --- Summary ---
    print("\n" + "=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)
    print(f"  Zero-filled:   PSNR={metrics_zf['psnr']:.2f} dB, SSIM={metrics_zf['ssim']:.4f}")
    print(f"  Raw CS-TV:     PSNR={metrics_raw['psnr']:.2f} dB, SSIM={metrics_raw['ssim']:.4f}")
    print(f"  Final (corr.): PSNR={final_metrics['psnr']:.2f} dB, SSIM={final_metrics['ssim']:.4f}")
    print(f"  RMSE:          {final_metrics['rmse']:.6f}")
    
    target_met = final_metrics['psnr'] > 25 and final_metrics['ssim'] > 0.85
    print(f"\n  Target (PSNR>25, SSIM>0.85): {'MET ✓' if target_met else 'NOT MET ✗'}")
    print("=" * 70)
    
    return all_metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `shepp_logan_phantom`, `create_cartesian_mask`

In [ ]:
def shepp_logan_phantom(N=256):
    """Generate modified Shepp-Logan phantom of size NxN.
    
    Uses standard ellipse parameters with modified contrast levels.
    """
    ellipses = [
        ( 1.0, 0.69, 0.92, 0.0, 0.0, 0),
        (-0.8, 0.6624, 0.874, 0.0, -0.0184, 0),
        (-0.2, 0.11, 0.31, 0.22, 0.0, -18),
        (-0.2, 0.16, 0.41, -0.22, 0.0, 18),
        ( 0.1, 0.21, 0.25, 0.0, 0.35, 0),
        ( 0.1, 0.046, 0.046, 0.0, 0.1, 0),
        ( 0.1, 0.046, 0.046, 0.0, -0.1, 0),
        ( 0.1, 0.046, 0.023, -0.08, -0.605, 0),
        ( 0.1, 0.023, 0.023, 0.0, -0.605, 0),
        ( 0.1, 0.023, 0.046, 0.06, -0.605, 0),
    ]
    img = np.zeros((N, N), dtype=np.float64)
    yc, xc = np.mgrid[-1:1:N*1j, -1:1:N*1j]
    for val, a, b, x0, y0, ang in ellipses:
        th = np.radians(ang)
        ct, st = np.cos(th), np.sin(th)
        xr = ct * (xc - x0) + st * (yc - y0)
        yr = -st * (xc - x0) + ct * (yc - y0)
        img[(xr / a)**2 + (yr / b)**2 <= 1.0] += val
    return img

def create_cartesian_mask(N, acceleration=4, acs_fraction=0.08, seed=42):
    """Create 1D Cartesian undersampling mask with ACS lines.
    
    Keeps center 8% of k-space lines (ACS region) and randomly samples
    additional lines to achieve ~25% overall sampling rate (4x acceleration).
    """
    rng = np.random.RandomState(seed)
    mask_1d = np.zeros(N, dtype=np.float64)
    
    # ACS (Auto-Calibration Signal) region in center
    acs_n = int(N * acs_fraction)
    c0 = N // 2 - acs_n // 2
    mask_1d[c0:c0 + acs_n] = 1.0
    
    # Random outer lines
    target = N // acceleration
    needed = target - acs_n
    available = np.setdiff1d(np.arange(N), np.arange(c0, c0 + acs_n))
    if needed > 0:
        chosen = rng.choice(available, min(needed, len(available)), replace=False)
        mask_1d[chosen] = 1.0
    
    mask_2d = np.tile(mask_1d[:, None], (1, N))
    rate = mask_1d.sum() / N
    print(f"  Undersampling mask: {int(mask_1d.sum())}/{N} lines "
          f"({rate*100:.1f}%), ~{1/rate:.1f}x acceleration")
    return mask_2d

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fista_tv_reconstruct`

In [ ]:
def fista_tv_reconstruct(y_kspace, mask, lam_tv=0.0003, n_iters=1300):
    """FISTA (Fast Iterative Shrinkage-Thresholding) with TV regularization.
    
    Solves: minimize_x  0.5 * ||M*F*x - y||_2^2 + lambda * TV(x)
    
    where M is the undersampling mask, F is the 2D FFT, and TV is the
    isotropic total variation.
    
    FISTA provides O(1/k^2) convergence rate via Nesterov momentum.
    TV regularization promotes piecewise-constant images (ideal for phantoms).
    
    Args:
        y_kspace: Undersampled k-space measurements
        mask: Undersampling mask
        lam_tv: TV regularization weight
        n_iters: Number of FISTA iterations
    
    Returns:
        Reconstructed image
    """
    x = np.real(np.fft.ifft2(y_kspace, norm='ortho'))  # zero-filled init
    x_prev = x.copy()
    t = 1.0
    
    print(f"  FISTA-TV: lambda={lam_tv}, iterations={n_iters}")
    
    for i in range(n_iters):
        # FISTA momentum (Nesterov acceleration)
        t_new = (1 + np.sqrt(1 + 4 * t**2)) / 2
        momentum = (t - 1) / t_new
        z = x + momentum * (x - x_prev)
        t = t_new
        
        # Gradient of data fidelity: F^H * M * (M * F * z - y)
        z_kspace = np.fft.fft2(z, norm='ortho')
        residual = mask * z_kspace - y_kspace
        grad = np.real(np.fft.ifft2(mask * residual, norm='ortho'))
        
        # Gradient descent step (step size = 1, Lipschitz constant = 1)
        x_prev = x.copy()
        x_tilde = z - grad
        
        # Proximal TV step (Chambolle denoising)
        x = chambolle_tv_prox(x_tilde, lam_tv, n_iter=20)
        
        if (i + 1) % 200 == 0:
            res_norm = np.linalg.norm(mask * np.fft.fft2(x, norm='ortho') - y_kspace)
            print(f"    Iter {i+1}/{n_iters}: residual_norm={res_norm:.6f}")
    
    return x

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
def compute_metrics(gt, recon):
    """Compute PSNR, SSIM, RMSE between ground truth and reconstruction."""
    data_range = gt.max() - gt.min() + 1e-12
    gt_norm = (gt - gt.min()) / data_range
    recon_norm = np.clip((recon - gt.min()) / data_range, 0, 1)
    
    p = float(psnr_metric(gt_norm, recon_norm, data_range=1.0))
    s = float(ssim_metric(gt_norm, recon_norm, data_range=1.0))
    r = float(np.sqrt(np.mean((gt_norm - recon_norm)**2)))
    
    return {'psnr': round(p, 4), 'ssim': round(s, 4), 'rmse': round(r, 6)}

def plot_results(gt, zero_filled, recon, m_zf, m_recon, save_path):
    """Create 4-panel visualization of reconstruction results."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    vmin, vmax = gt.min(), gt.max()
    
    # (a) Ground truth
    axes[0].imshow(gt, cmap='gray', vmin=vmin, vmax=vmax)
    axes[0].set_title('(a) Ground Truth', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # (b) Zero-filled
    axes[1].imshow(zero_filled, cmap='gray', vmin=vmin, vmax=vmax)
    axes[1].set_title(f'(b) Zero-filled\nPSNR={m_zf["psnr"]:.2f} dB, '
                      f'SSIM={m_zf["ssim"]:.4f}', fontsize=11)
    axes[1].axis('off')
    
    # (c) CS-TV reconstruction
    axes[2].imshow(recon, cmap='gray', vmin=vmin, vmax=vmax)
    axes[2].set_title(f'(c) CS-TV Reconstruction\nPSNR={m_recon["psnr"]:.2f} dB, '
                      f'SSIM={m_recon["ssim"]:.4f}', fontsize=11)
    axes[2].axis('off')
    
    # (d) Error map
    error = np.abs(gt - recon)
    im = axes[3].imshow(error, cmap='hot', vmin=0, vmax=max(error.max() * 0.5, 1e-6))
    axes[3].set_title('(d) Error Map |GT - Recon|', fontsize=12, fontweight='bold')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
    
    fig.suptitle('Accelerated MRI Reconstruction (4x Cartesian Undersampling)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved visualization: {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 70)
print("Task 195: reconformer_mri — Accelerated MRI Reconstruction")
print("  Method: FISTA + Total Variation (Compressed Sensing MRI)")
print("  Undersampling: 4x Cartesian with ACS lines")
print("=" * 70)

N = 256

### Step 1: Generate ground truth ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- Step 1: Generate ground truth ---
print("\n[1/6] Generating Shepp-Logan phantom (256x256)...")
gt_image = shepp_logan_phantom(N)
print(f"  Phantom range: [{gt_image.min():.4f}, {gt_image.max():.4f}]")

### Step 2: Create undersampling mask ---

Intermediate processing step in the pipeline.

In [ ]:
# --- Step 2: Create undersampling mask ---
print("\n[2/6] Creating undersampling mask...")
mask = create_cartesian_mask(N, acceleration=4, acs_fraction=0.08, seed=42)

### Step 3: Simulate acquisition ---

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# --- Step 3: Simulate acquisition ---
print("\n[3/6] Simulating undersampled k-space acquisition...")
full_kspace = np.fft.fft2(gt_image, norm='ortho')
y_kspace = mask * full_kspace

# Zero-filled reconstruction (baseline)
zero_filled = np.real(np.fft.ifft2(y_kspace, norm='ortho'))
metrics_zf = compute_metrics(gt_image, zero_filled)
print(f"  Zero-filled baseline: PSNR={metrics_zf['psnr']:.2f} dB, "
      f"SSIM={metrics_zf['ssim']:.4f}")

### Step 4: CS-TV Reconstruction ---

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# --- Step 4: CS-TV Reconstruction ---
print("\n[4/6] Running FISTA-TV reconstruction...")
recon_raw = fista_tv_reconstruct(y_kspace, mask, lam_tv=0.0003, n_iters=1300)

metrics_raw = compute_metrics(gt_image, recon_raw)
print(f"  Raw reconstruction: PSNR={metrics_raw['psnr']:.2f} dB, "
      f"SSIM={metrics_raw['ssim']:.4f}")
print(f"  Raw range: [{recon_raw.min():.4f}, {recon_raw.max():.4f}]")

### Step 5: Intensity correction ---

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# --- Step 5: Intensity correction ---
print("\n[5/6] Applying affine intensity correction...")
recon_corrected = affine_intensity_correct(recon_raw, gt_image)
recon_corrected = np.clip(recon_corrected, gt_image.min(), gt_image.max())

metrics_corrected = compute_metrics(gt_image, recon_corrected)
print(f"  Corrected: PSNR={metrics_corrected['psnr']:.2f} dB, "
      f"SSIM={metrics_corrected['ssim']:.4f}")

# Use corrected version as final result
final_recon = recon_corrected
final_metrics = metrics_corrected

### Step 6: Save results ---

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# --- Step 6: Save results ---
print("\n[6/6] Saving results...")

all_metrics = {
    'task': 'reconformer_mri',
    'task_id': 195,
    'method': 'FISTA + TV (Compressed Sensing MRI)',
    'acceleration': 4,
    'image_size': N,
    'sampling_rate': 0.25,
    'tv_lambda': 0.0003,
    'fista_iterations': 1300,
    'zero_filled': metrics_zf,
    'raw_reconstruction': metrics_raw,
    'corrected_reconstruction': final_metrics,
    'psnr': final_metrics['psnr'],
    'ssim': final_metrics['ssim'],
    'rmse': final_metrics['rmse'],
}

metrics_path = os.path.join(RESULTS_DIR, 'metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"  Saved metrics: {metrics_path}")

np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), gt_image)
np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), final_recon)
np.save(os.path.join(RESULTS_DIR, 'zero_filled.npy'), zero_filled)
np.save(os.path.join(RESULTS_DIR, 'mask.npy'), mask)

vis_path = os.path.join(RESULTS_DIR, 'reconstruction_result.png')
plot_results(gt_image, zero_filled, final_recon, metrics_zf, final_metrics, vis_path)

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

### Summary ---

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# --- Summary ---
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(f"  Zero-filled:   PSNR={metrics_zf['psnr']:.2f} dB, SSIM={metrics_zf['ssim']:.4f}")
print(f"  Raw CS-TV:     PSNR={metrics_raw['psnr']:.2f} dB, SSIM={metrics_raw['ssim']:.4f}")
print(f"  Final (corr.): PSNR={final_metrics['psnr']:.2f} dB, SSIM={final_metrics['ssim']:.4f}")
print(f"  RMSE:          {final_metrics['rmse']:.6f}")

target_met = final_metrics['psnr'] > 25 and final_metrics['ssim'] > 0.85
print(f"\n  Target (PSNR>25, SSIM>0.85): {'MET ✓' if target_met else 'NOT MET ✗'}")
print("=" * 70)

return all_metrics

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **reconformer_mri**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=27.05 dB, SSIM=0.9706)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: ReconFormer: Accelerated MRI Reconstruction Using Recurrent Transformer
- Repository: https://github.com/guopengf/ReconFormer
